# 00 - Preparación de datos en el lakehouse.
### Crear tablas Delta
Ejecuta estas celdas para crear tablas Delta a partir de datos en bruto.

#### Celda - Configuración de sesión Spark
Esta celda habilita dos características de Fabric que optimizan cómo se escriben y leen los datos en las celdas posteriores. V-order optimiza el diseño de parquet para lecturas más rápidas y mejor compresión. Optimize Write reduce el número de archivos escritos y aumenta el tamaño de los archivos individuales.

In [ ]:
# Configuración de Spark para optimizar escritura y lectura en parquet/delta
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize", "1073741824")

#### Celda - Fact Sale
Esta celda lee datos parquet en bruto desde `Files/bronze/fact_sale_1y_full`, agrega columnas de particiones de fecha (`Year`, `Quarter` y `Month`), y escribe `fact_sale` como una tabla Delta particionada por `Year` y `Quarter`.

In [ ]:
# Importar funciones necesarias de PySpark
from pyspark.sql.functions import col, year, month, quarter

# Leer datos parquet de ventas desde la capa bronze
df = spark.read.format("parquet").load("Files/bronze/fact_sale_1y_full")

# Agregar columnas de partición de fecha extraídas de InvoiceDateKey
df = df.withColumn("Year", year(col("InvoiceDateKey")))
df = df.withColumn("Quarter", quarter(col("InvoiceDateKey")))
df = df.withColumn("Month", month(col("InvoiceDateKey")))

# Escribir la tabla fact_sale en formato Delta, particionada por Year y Quarter
df.write.mode("overwrite").format("delta").partitionBy("Year", "Quarter").save("Tables/silver/fact_sale")

#### Celda - Dimensiones
Esta celda lee los cinco conjuntos de datos parquet de dimensiones y los escribe como tablas Delta (`dimension_city`, `dimension_customer`, `dimension_date`, `dimension_employee` y `dimension_stock_item`) bajo `Tables/silver/...`.

In [ ]:
# Función para cargar datos desde la capa bronze y escribirlos en la capa silver
def load_full_data_from_source(table_name):
    # Leer datos en formato parquet desde la carpeta bronze
    df = spark.read.format("parquet").load("Files/bronze/" + table_name)
    
    # Eliminar la columna Photo si existe (para reducir tamaño de datos)
    if "Photo" in df.columns:
        df = df.drop("Photo")
    
    # Escribir los datos en formato Delta en la capa silver
    df.write.mode("overwrite").format("delta").save("Tables/silver/" + table_name)

# Lista de tablas de dimensiones a procesar
full_tables = [
    "dimension_city",
    "dimension_customer",
    "dimension_date",
    "dimension_employee",
    "dimension_stock_item",
]

# Procesar cada tabla de dimensión
for table in full_tables:
    load_full_data_from_source(table)